In [ ]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
contacts = pd.read_csv('../data/task01/contacts.csv')
customers = pd.read_csv('../data/task01/customers.csv')
purchases = pd.read_csv('../data/task01/purchases.csv')


In [ ]:
print('Contacts table shape:', contacts.shape)
print('Customers table shape:', customers.shape)
print('Purchases table shape:', purchases.shape)

display(contacts.head())
display(customers.head())
display(purchases.head())


In [ ]:
# Removing unnamed columns in the customers table

customers = customers.loc[:, ~customers.columns.str.contains('^Unnamed')]

customers.head()


In [ ]:
# Tables info

print("Contacts table info:\n")
print(contacts.info())
print("\nCustomers table info:\n")
print(customers.info())

print("\nPurchases table info:\n")
print(purchases.info())

In [ ]:
# Parsing date fields

customers["signup_date"] = pd.to_datetime(
    customers["signup_date"],
    errors="coerce",
    dayfirst=True
)

purchases["purchase_date"] = pd.to_datetime(
    purchases["purchase_date"],
    errors="coerce",
    dayfirst=True
)

print("\nCustomers table info:")
print(customers.info())

print("\nPurchases table info:")
print(purchases.info())

In [ ]:
# Missing values
print(f"Missing values in customers: \n{customers.isna().sum()}")
print()
print(f"Missing values in contacts: \n{contacts.isna().sum()}")
print()
print(f"Missing values in purchases: \n{purchases.isna().sum()}")


In [ ]:
# Filling missing values

customers["segment"] = customers["segment"].fillna("Unknown")
customers["city"] = customers["city"].fillna("Unknown")

In [ ]:
# Removing rows

purchases_clean = purchases[
    purchases["brand"].notna()
    & purchases["purchase_date"].notna()
    & purchases["amount"].notna()
    & (purchases["amount"] > 0)
].copy()

print(f"Original purchases rows: {len(purchases)}")
print(f"Clean purchases rows: {len(purchases_clean)}")
print(f"Rows removed: {len(purchases) - len(purchases_clean)}")

print(f"\nMissing values in customers after cleaning:\n{customers.isna().sum()}")

print(f"\nMissing values in purchases_clean:\n{purchases_clean.isna().sum()}")

print(f"\nInvalid purchase amounts remaining:\n{purchases_clean[purchases_clean['amount'] <= 0]}")

In [ ]:
# Merge contacts with customers

customer_base = customers.merge(
    contacts,
    on="customer_id",
    how="left"
)

print(f"Customers shape: {customers.shape}")
print(f"Contacts shape: {contacts.shape}")
print(f"Customer base shape: {customer_base.shape}")

print(f"\nMissing values in customer_base:\n{customer_base.isna().sum()}")

print("\nFirst 5 rows:")
print(customer_base.head())

In [ ]:
# Merge purchases with customer

purchases_enriched = purchases_clean.merge(
    customer_base,
    on="customer_id",
    how="left"
)

print(f"Purchases clean shape: {purchases_clean.shape}")
print(f"Purchases enriched shape: {purchases_enriched.shape}")

print(f"\nMissing values in purchases_enriched:\n{purchases_enriched.isna().sum()}")

print("\nFirst 5 rows:")
print(purchases_enriched.head())

In [ ]:
# Customer analytic table

customer_analytics = (
    purchases_enriched
    .groupby("customer_id")
    .agg(
        total_spend=("amount", "sum"),
        average_spend=("amount", "mean"),
        purchase_count=("purchase_id", "count"),
        first_purchase=("purchase_date", "min"),
        last_purchase=("purchase_date", "max"),
        brands_bought=("brand", lambda x: ", ".join(sorted(x.unique()))),
        channels_used=("channel", lambda x: ", ".join(sorted(x.unique())))
    )
    .reset_index()
)

customer_analytics = customer_analytics.merge(
    customer_base,
    on="customer_id",
    how="right"
)

customer_analytics["total_spend"] = customer_analytics["total_spend"].fillna(0)
customer_analytics["average_spend"] = customer_analytics["average_spend"].fillna(0)
customer_analytics["purchase_count"] = customer_analytics["purchase_count"].fillna(0)

print(f"Customer analytics shape: {customer_analytics.shape}")
print(f"\nMissing values in customer_analytics:\n{customer_analytics.isna().sum()}")
print("\nFirst 5 rows:")
print(customer_analytics.head())

In [ ]:
# Replacing missing brands_bought and channels_used with None

customer_analytics["brands_bought"] = customer_analytics["brands_bought"].fillna("None")
customer_analytics["channels_used"] = customer_analytics["channels_used"].fillna("None")

print(f"Missing values in customer_analytics after tidy:\n{customer_analytics.isna().sum()}")
print("\nFirst 5 rows:")
print(customer_analytics.head())

In [ ]:
# Total and average spend per customer

spend_per_customer = (
    purchases_enriched
    .groupby("customer_id")
    .agg(
        total_spend=("amount", "sum"),
        average_spend=("amount", "mean"),
        purchase_count=("purchase_id", "count")
    )
    .reset_index()
    .sort_values("total_spend", ascending=False)
)

print(f"Spend per customer shape: {spend_per_customer.shape}")
print(spend_per_customer.head(10))

In [ ]:
# Total and average spend per group

spend_per_group = (
    purchases_enriched
    .groupby("segment")
    .agg(
        total_spend=("amount", "sum"),
        average_spend=("amount", "mean"),
        unique_customers=("customer_id", "nunique"),
        purchase_count=("purchase_id", "count")
    )
    .reset_index()
    .sort_values("total_spend", ascending=False)
)

print(f"Spend per group shape: {spend_per_group.shape}")
print(spend_per_group)

In [ ]:
# Investigate suspicious segment value

print("Unique customer segments:")
print(customers["segment"].unique())

print("\nRows where segment is '3':")
print(customers[customers["segment"].astype(str) == "3"])

print(f"\nSegment counts: {customers["segment"].value_counts(dropna=False)}")


In [ ]:
# Fixing invalid segment value

customers["segment"] = customers["segment"].replace("3", "Unknown")
customer_base["segment"] = customer_base["segment"].replace("3", "Unknown")
purchases_enriched["segment"] = purchases_enriched["segment"].replace("3", "Unknown")
customer_analytics["segment"] = customer_analytics["segment"].replace("3", "Unknown")

In [ ]:
spend_per_group = (
    purchases_enriched
    .groupby("segment")
    .agg(
        total_spend=("amount", "sum"),
        average_spend=("amount", "mean"),
        unique_customers=("customer_id", "nunique"),
        purchase_count=("purchase_id", "count")
    )
    .reset_index()
    .sort_values("total_spend", ascending=False)
)

print(f"Spend per group shape: {spend_per_group.shape}")
print(spend_per_group)

In [ ]:
# Total spend per brand and channel

spend_per_brand_channel = (
    purchases_enriched
    .groupby(["brand", "channel"])
    .agg(
        total_spend=("amount", "sum"),
        average_spend=("amount", "mean"),
        unique_customers=("customer_id", "nunique"),
        purchase_count=("purchase_id", "count")
    )
    .reset_index()
    .sort_values(["brand", "channel"])
)

print(f"Spend per brand and channel shape: {spend_per_brand_channel.shape}")
print(spend_per_brand_channel)

In [ ]:
# Online vs offline share per brand

brand_totals = (
    spend_per_brand_channel
    .groupby("brand")["total_spend"]
    .sum()
    .reset_index()
    .rename(columns={"total_spend": "brand_total_spend"})
)

brand_channel_share = spend_per_brand_channel.merge(
    brand_totals,
    on="brand",
    how="left"
)

brand_channel_share["channel_share"] = (
    brand_channel_share["total_spend"] / brand_channel_share["brand_total_spend"]
)

brand_channel_share["channel_share_pct"] = (
    brand_channel_share["channel_share"] * 100
).round(2)

print(f"Brand channel share shape: {brand_channel_share.shape}")
print(brand_channel_share)

In [ ]:
online_offline_share = (
    brand_channel_share
    .pivot_table(
        index="brand",
        columns="channel",
        values="channel_share_pct",
        fill_value=0
    )
    .reset_index()
)

print("\nOnline vs Offline Share by Brand:")
print(online_offline_share)

In [ ]:
# Best city per brand

brand_city_spend = (
    purchases_enriched
    .groupby(["brand", "city"])
    .agg(
        total_spend=("amount", "sum"),
        average_spend=("amount", "mean"),
        unique_customers=("customer_id", "nunique"),
        purchase_count=("purchase_id", "count")
    )
    .reset_index()
    .sort_values(["brand", "total_spend"], ascending=[True, False])
)

print(f"Brand city spend shape: {brand_city_spend.shape}")
print(brand_city_spend)

In [ ]:
# Best cities per brand

best_city_per_brand = (
    brand_city_spend
    .sort_values(["brand", "total_spend"], ascending=[True, False])
    .groupby("brand")
    .head(1)
    .reset_index(drop=True)
)

print("\nBest city per brand:")
print(best_city_per_brand)

In [ ]:
# Spend per group segment plot

spend_per_group_plot = spend_per_group.sort_values("total_spend", ascending=False)

plt.bar(
    spend_per_group_plot["segment"],
    spend_per_group_plot["total_spend"]
)

plt.title("Total Spend per Customer Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Total Spend")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Brand info with customer information

customer_analytics

In [ ]:
# Existing/Potential brand customers db creation
brand_customer_db_list = []

for brand in brand_totals.brand:
    target_segments = purchases_enriched.loc[
        purchases_enriched["brand"] == brand,
        "segment"
    ].unique()

    existing_customers_id = purchases_enriched.loc[
        purchases_enriched["brand"] == brand,
        'customer_id'
    ]

    brand_customers = customer_base.loc[
        customer_base['segment'].isin(target_segments),
    ].copy()

    brand_customers['brand'] = brand

    brand_customers['customer_status'] = np.where(
        brand_customers['customer_id'].isin(existing_customers_id),
        'existing',
        'potential'
    )

    brand_customer_db_list.append(brand_customers)

brand_customer_db = pd.concat(brand_customer_db_list, ignore_index=True)

print(brand_customer_db.head())
print(brand_customer_db.shape)

In [ ]:
# Brand-Customer DB final

brand_customer_db['full_name'] = (
    brand_customer_db.first_name + " " + brand_customer_db.last_name
)

brand_customer_db_final = brand_customer_db[
    ['brand',
    'customer_status',
    'customer_id',
    'full_name',
    'email',
    'phone',
    'segment',
    'city'
     ]
]. copy()

print(brand_customer_db_final.head())

In [ ]:
# Brand exist vs potential customers

brand_customers_status_summary = (
    brand_customer_db_final
    .groupby(["brand", "customer_status"])
    .agg(customets=("customer_id", "nunique"))
    .reset_index()
)

print(brand_customers_status_summary)

In [ ]:
# Export all brand data to csv
brand_customer_db_final.to_csv(
    '../outputs/brand_customer_db_final.csv',
    index=False
)

In [ ]:
# Export csv per brand

for brand in brand_customer_db_final.brand.unique():
    brand_db = brand_customer_db_final[
        brand_customer_db_final.brand == brand
    ]

    brand_db.to_csv(
        f'../outputs/{brand}_db.csv',
    )

    print(f"{brand}_db.csv Created!")

In [ ]:
# Spend for Brand

purchases_enriched


In [ ]:
purchases_enriched['month'] = (
    purchases_enriched['purchase_date'].dt.to_period('M').dt.to_timestamp()
)

purchases_enriched.info()

In [ ]:
purchases_enriched['month_number'] = purchases_enriched.purchase_date.dt.month
purchases_enriched['month'] = purchases_enriched.purchase_date.dt.month_name()

brand_monthly_spend = (
    purchases_enriched
    .groupby(["brand", "month", 'month_number'])
    .agg(total_spend=("amount", "sum"))
    .reset_index()
    .sort_values(["month_number", "brand"])
)

print(brand_monthly_spend)

In [ ]:
# Brand Spend over time plot
for brand in brand_monthly_spend["brand"].unique():
    brand_data = brand_monthly_spend[
        brand_monthly_spend["brand"] == brand
    ].sort_values("month_number")

    plt.plot(
        brand_data["month"],
        brand_data["total_spend"],
        marker="o",
        linewidth=2,
        label=brand
    )

plt.title("Brand Spend Over Time")
plt.xlabel("Month")
plt.ylabel("Total Spend")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Brand")
plt.tight_layout()
plt.show()